In [ ]:
# =============================================================================
# Short Data Repository
# =============================================================================
#
# KRX 전 종목(KOSPI + KOSDAQ ~2,500개)의 공매도 데이터를 종목별 CSV로 관리.
#
# 저장 구조:
#   short_data/
#     005930.csv   ← 삼성전자 (한 파일 = 한 종목, 한 행 = 하루)
#     000660.csv   ← SK하이닉스
#     ...          ← ~2,500개 파일
#
# 각 CSV 컬럼:
#   date         거래일 YYYYMMDD
#   ─── pykrx short_vol ───
#   공매도        공매도 거래량 (주)
#   매수          총 매수 거래량 (주)
#   공매도비중     공매도 / 매수 비율 (%)
#   ─── pykrx short_bal (주의: ~3영업일 지연 공시) ───
#   공매도잔고     미결제 공매도 수량 (주)
#   상장주식수     발행주식수
#   공매도금액     공매도잔고 × 가격 (KRW)
#   시가총액       시가총액 (KRW)
#   잔고비중       공매도잔고 / 상장주식수 (%)
#   ─── Bloomberg (optional) ───
#   px_last       종가
#   free_float    유동주식 비율 (%)
#   index_px      소속 지수(코스피/코스닥) 종가
#
# 사용법:
#   1) 처음: backfill()  — 2021-05-03부터 전체 다운로드 (수 시간 소요)
#   2) 매일: daily_update() — 빠진 거래일만 추가 (~15초)
# =============================================================================

import logging
import sys
import time
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# ── pykrx 인증 ──
# KRX 데이터포털이 2024년 말부터 로그인 필수로 변경됨.
# krx_auth.py가 pykrx 내부의 HTTP 요청을 인증된 세션으로 패치함.
sys.path.insert(0, str(Path.cwd() / "AI" / "kr_disclosure"))
sys.path.insert(0, str(Path.cwd() / "AI" / "email_analysis"))
sys.path.insert(0, str(Path.cwd() / "AI" / "emailer"))

from krx_auth import init_krx_auth
from pykrx import stock
init_krx_auth()

# ── Bloomberg (없어도 동작 — 해당 컬럼만 NaN) ──
try:
    from xbbg import blp
    _HAS_BBG = True
except ImportError:
    _HAS_BBG = False

# ── 설정값 ──
SHORT_START = date(2021, 5, 3)   # 공매도 금지 해제일 (이 날부터 데이터 존재)
PYKRX_SLEEP = 0.35               # KRX API rate limit 대비 호출 간격 (초)
BBG_BATCH   = 200                # Bloomberg bdh 한 번에 보낼 수 있는 최대 종목 수
REPO_DIR    = Path("short_data") # CSV 파일 저장 위치
REF_TICKER  = "005930"           # 삼성전자 — 거래일 판별 기준 (절대 상폐 안 됨)

# Bloomberg 지수 티커 매핑
_IDX_MAP = {"KOSPI": "KOSPI Index", "KOSDAQ": "KOSDAQ Index"}

# pykrx 반환 컬럼명 (한국어)
VOL_COLS = ["공매도", "매수", "공매도비중"]          # short_vol에서 가져올 컬럼
BAL_COLS = ["공매도잔고", "상장주식수", "공매도금액",  # short_bal에서 가져올 컬럼
            "시가총액", "잔고비중"]

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)-7s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("short_repo")

print(f"Repo:      {REPO_DIR.resolve()}")
print(f"Bloomberg: {'available' if _HAS_BBG else 'not installed'}")

In [ ]:
# =============================================================================
# pykrx Bulk Fetchers
# =============================================================================
#
# 핵심 최적화:
#   get_shorting_status_by_date(start, end, ticker)  → 종목당 1 call → 2500 call → ~3시간
#   get_shorting_volume_by_ticker(date, market)       → 시장당 1 call → 4 call   → ~3초
#
# 이 노트북은 후자(bulk API)를 사용. 하루치 전 종목 데이터를 4번의 API call로 가져옴.
# =============================================================================

def fetch_short_vol(date_str: str) -> pd.DataFrame | None:
    """하루치 전 종목 공매도 거래량 (KOSPI + KOSDAQ).

    pykrx의 get_shorting_volume_by_ticker()를 KOSPI/KOSDAQ 각 1번씩 호출.
    휴장일이면 None 반환 (pykrx가 모든 값이 0인 DataFrame을 돌려줌).
    """
    frames = []
    for market in ("KOSPI", "KOSDAQ"):
        try:
            df = stock.get_shorting_volume_by_ticker(date_str, market)

            # pykrx는 휴장일에도 빈 데이터 대신 모든 값이 0인 DataFrame을 반환함
            # → 공매도 합계가 0이면 휴장일로 판단하고 skip
            if df is None or df.empty:
                continue
            if df.get("공매도", pd.Series()).sum() == 0:
                continue

            # pykrx는 티커를 index로 반환 → 컬럼으로 변환
            df = df.copy()
            df.index.name = "티커"
            df = df.reset_index()
            df["티커"] = df["티커"].astype(str).str.zfill(6)  # 6자리 zero-pad
            df["시장"] = market
            frames.append(df)
        except Exception as e:
            log.warning("short_vol %s %s: %s", market, date_str, e)

        # KRX API rate limit 방지 (너무 빠르면 차단당함)
        time.sleep(PYKRX_SLEEP)

    return pd.concat(frames, ignore_index=True) if frames else None


def fetch_short_bal(date_str: str) -> pd.DataFrame | None:
    """하루치 전 종목 공매도 잔고 (KOSPI + KOSDAQ).

    주의: 잔고 데이터는 거래일 기준 ~3영업일 지연 공시됨.
    예) 월요일 잔고 → 목요일 오후에 공시
    따라서 short_vol과 동일 날짜에 데이터가 없을 수 있음 (정상).
    """
    frames = []
    for market in ("KOSPI", "KOSDAQ"):
        try:
            df = stock.get_shorting_balance_by_ticker(date_str, market)
            if df is None or df.empty:
                continue
            df = df.copy()
            df.index.name = "티커"
            df = df.reset_index()
            df["티커"] = df["티커"].astype(str).str.zfill(6)
            df["시장"] = market
            frames.append(df)
        except Exception as e:
            log.warning("short_bal %s %s: %s", market, date_str, e)
        time.sleep(PYKRX_SLEEP)

    return pd.concat(frames, ignore_index=True) if frames else None


def merge_vol_bal(vol_df, bal_df, date_str: str) -> pd.DataFrame:
    """한 날짜의 short_vol과 short_bal을 종목 기준으로 합침.

    - vol과 bal 모두 있으면: outer join (한쪽만 있는 종목도 포함)
    - 한쪽만 있으면: 그것만 사용
    - 둘 다 없으면: 빈 DataFrame 반환

    pykrx가 "비중"이라는 동일한 컬럼명을 두 데이터셋에서 사용하므로
    vol → "공매도비중", bal → "잔고비중"으로 rename해서 충돌 방지.
    """
    if vol_df is None and bal_df is None:
        return pd.DataFrame()

    if vol_df is not None:
        vol_df = vol_df.rename(columns={"비중": "공매도비중"})
        vol_df = vol_df[["티커", "시장"] + VOL_COLS]
    if bal_df is not None:
        bal_df = bal_df.rename(columns={"비중": "잔고비중"})
        bal_df = bal_df[["티커", "시장"] + BAL_COLS]

    if vol_df is not None and bal_df is not None:
        merged = pd.merge(vol_df, bal_df, on=["티커", "시장"], how="outer")
    elif vol_df is not None:
        merged = vol_df
    else:
        merged = bal_df

    merged.insert(0, "date", date_str)  # 날짜 컬럼을 맨 앞에 삽입
    return merged

In [ ]:
# =============================================================================
# Bloomberg Enrichment (optional — 없어도 pykrx 데이터만으로 동작함)
# =============================================================================
#
# Bloomberg Terminal이 있으면 3가지 필드를 추가:
#   px_last    — 종가 (pykrx에서는 공매도 데이터만 줌, 종가는 별도)
#   free_float — 유동주식 비율 (%) (공매도 잔고의 실질적 비중 계산에 필요)
#   index_px   — 소속 지수 종가 (KOSPI 종목이면 코스피지수, KOSDAQ이면 코스닥지수)
#
# Bloomberg 없이 실행하면 이 3개 컬럼이 NaN으로 채워짐.
# =============================================================================

def fetch_bbg_daily(krx_tickers: list[str], d: date) -> tuple[dict, dict, dict]:
    """하루치 Bloomberg 데이터 fetch. (px_map, ff_map, idx_map) 반환.
    Bloomberg 없으면 빈 dict 3개 반환 — 호출하는 쪽에서 .map()하면 NaN이 들어감.
    """
    if not _HAS_BBG or not krx_tickers:
        return {}, {}, {}

    px_map, ff_map, idx_map = {}, {}, {}
    date_s = d.strftime("%Y-%m-%d")

    # 개별 종목 데이터 — 200개씩 배치 (Bloomberg API 제한)
    for i in range(0, len(krx_tickers), BBG_BATCH):
        batch = krx_tickers[i:i + BBG_BATCH]
        bbg_batch = [f"{t} KS Equity" for t in batch]  # "005930" → "005930 KS Equity"
        try:
            hist = blp.bdh(bbg_batch, ["PX_LAST", "EQY_FREE_FLOAT_PCT"], date_s, date_s)
            for krx, bbg in zip(batch, bbg_batch):
                for field, dest in [("PX_LAST", px_map), ("EQY_FREE_FLOAT_PCT", ff_map)]:
                    try:
                        val = hist[(bbg, field)].dropna()
                        if not val.empty:
                            dest[krx] = float(val.iloc[0])
                    except KeyError:
                        pass  # Bloomberg에 없는 종목 (상폐, ETF 등)
        except Exception as e:
            log.warning("Bloomberg batch failed: %s", e)

    # 지수 데이터 — 코스피 + 코스닥 종가
    try:
        idx_hist = blp.bdh(list(_IDX_MAP.values()), "PX_LAST", date_s, date_s)
        for market, bbg_idx in _IDX_MAP.items():
            try:
                val = idx_hist[(bbg_idx, "PX_LAST")].dropna()
                if not val.empty:
                    idx_map[market] = float(val.iloc[0])
            except KeyError:
                pass
    except Exception as e:
        log.warning("Bloomberg index failed: %s", e)

    return px_map, ff_map, idx_map


def fetch_bbg_history(krx_tickers: list[str], start: date, end: date):
    """기간 전체 Bloomberg 히스토리. backfill 시 사용.
    Returns (px_hist, ff_hist, idx_hist).
    px_hist = {ticker: Series(YYYYMMDD → price)}, ff_hist 동일 구조.
    """
    if not _HAS_BBG:
        return {}, {}, {}

    start_s, end_s = start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")
    px_result, ff_result = {}, {}

    for i in range(0, len(krx_tickers), BBG_BATCH):
        batch = krx_tickers[i:i + BBG_BATCH]
        bbg_batch = [f"{t} KS Equity" for t in batch]
        log.info("  Bloomberg batch %d-%d / %d",
                 i + 1, min(i + BBG_BATCH, len(krx_tickers)), len(krx_tickers))
        try:
            hist = blp.bdh(bbg_batch, ["PX_LAST", "EQY_FREE_FLOAT_PCT"], start_s, end_s)
            for krx, bbg in zip(batch, bbg_batch):
                for field, dest in [("PX_LAST", px_result), ("EQY_FREE_FLOAT_PCT", ff_result)]:
                    try:
                        s = hist[(bbg, field)].dropna()
                        if not s.empty:
                            s.index = s.index.strftime("%Y%m%d")  # datetime → YYYYMMDD
                            dest[krx] = s
                    except KeyError:
                        pass
        except Exception as e:
            log.warning("  Bloomberg batch failed: %s", e)

    # 지수 히스토리
    idx_hist = {}
    try:
        ih = blp.bdh(list(_IDX_MAP.values()), "PX_LAST", start_s, end_s)
        for market, bbg_idx in _IDX_MAP.items():
            try:
                s = ih[(bbg_idx, "PX_LAST")].dropna()
                s.index = s.index.strftime("%Y%m%d")
                idx_hist[market] = s
            except KeyError:
                pass
    except Exception as e:
        log.warning("Bloomberg index history failed: %s", e)

    log.info("PX_LAST: %d, FREE_FLOAT: %d", len(px_result), len(ff_result))
    return px_result, ff_result, idx_hist

In [ ]:
# =============================================================================
# Per-Stock I/O & Gap Detection
# =============================================================================

def stock_path(ticker: str) -> Path:
    """종목 CSV 경로. 예: short_data/005930.csv"""
    return REPO_DIR / f"{ticker}.csv"


def read_stock(ticker: str) -> pd.DataFrame:
    """종목 CSV 읽기. 파일이 없거나 깨져 있으면 빈 DataFrame 반환."""
    p = stock_path(ticker)
    if not p.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(p, dtype={"date": str})
    except Exception:
        return pd.DataFrame()


def write_stock(ticker: str, df: pd.DataFrame) -> None:
    """종목 CSV 전체 덮어쓰기. 디렉토리 없으면 자동 생성."""
    p = stock_path(ticker)
    p.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(p, index=False, encoding="utf-8")


def latest_repo_date() -> str | None:
    """Repo에서 가장 최근 날짜를 조회.

    FIX: 원본은 sorted(glob)[:5]로 첫 5개 CSV만 읽었음.
    → 종목코드순이라 000040~000100 근처만 확인 → 상폐 종목이면 최신 날짜를 놓침.
    → 삼성전자(005930) + SK하이닉스(000660)를 기준으로 변경. 절대 상폐 안 되는 종목.
    """
    if not REPO_DIR.exists():
        return None
    latest = None
    for ref in ["005930", "000660"]:
        p = stock_path(ref)
        if not p.exists():
            continue
        try:
            df = pd.read_csv(p, usecols=["date"], dtype={"date": str})
            if not df.empty:
                max_d = df["date"].max()
                if latest is None or max_d > latest:
                    latest = max_d
        except Exception:
            continue
    return latest


def find_missing_trading_days(since: str, until: str | None = None) -> list[str]:
    """repo의 마지막 날짜 이후 ~ 오늘까지 빠진 실제 KRX 거래일 목록.

    FIX: 원본은 weekday(월~금)만 체크했음.
    → 설날, 추석, 근로자의 날 등 공휴일에도 불필요한 API call 발생.
    → pykrx의 get_market_ohlcv()를 이용해 실제 거래일만 조회.
      (이 함수는 KRX 개장일만 행을 반환하므로 공휴일이 자동으로 제외됨)
    """
    if until is None:
        until = date.today().strftime("%Y%m%d")

    # since는 이미 있는 날짜 → 그 다음 날부터 조회
    since_dt = date(int(since[:4]), int(since[4:6]), int(since[6:]))
    start = (since_dt + timedelta(days=1)).strftime("%Y%m%d")

    if start > until:
        return []

    # pykrx는 KRX 개장일만 DataFrame 행으로 반환함
    ohlcv = stock.get_market_ohlcv(start, until, REF_TICKER)
    if ohlcv is None or ohlcv.empty:
        return []

    return [d.strftime("%Y%m%d") for d in ohlcv.index]

In [ ]:
# =============================================================================
# Backfill — 최초 실행 시 2021-05-03부터 전체 히스토리 다운로드
# =============================================================================
#
# 흐름:
#   1) pykrx로 실제 거래일 캘린더 조회 (get_market_ohlcv → 삼성전자 기준)
#   2) 날짜별로 bulk API 호출 (4 calls/day × ~1200일 = ~4800 calls)
#   3) 날짜별 데이터를 종목별로 쪼개서 {ticker}.csv에 append
#   4) (Bloomberg 있으면) 종가/유동주식/지수가격 enrichment
#
# 소요 시간: ~3~4시간 (네트워크 속도에 따라 다름)
# 100일마다 중간 저장 (flush_to_stocks) → 중간에 끊겨도 다시 시작하면 이미 있는 날짜 skip
# =============================================================================

def flush_to_stocks(chunks: list[pd.DataFrame]) -> None:
    """축적된 날짜별 데이터를 종목별 CSV로 분산 저장.

    chunks = [날짜1의 전종목 DataFrame, 날짜2의 전종목 DataFrame, ...]
    이걸 종목 기준으로 groupby해서 각 종목 CSV에 append.
    이미 해당 날짜가 있는 종목은 중복 저장 안 함 (idempotent).
    """
    big = pd.concat(chunks, ignore_index=True)
    for ticker, group in big.groupby("티커"):
        group = group.drop(columns=["티커"]).sort_values("date")
        existing = read_stock(ticker)
        if existing.empty:
            # 새 종목 — 파일 생성
            write_stock(ticker, group)
        else:
            # 기존 종목 — 새 날짜만 append
            new_dates = set(group["date"]) - set(existing["date"])
            if new_dates:
                append = group[group["date"].isin(new_dates)]
                combined = pd.concat([existing, append], ignore_index=True).sort_values("date")
                write_stock(ticker, combined)


def backfill(start_date: date = SHORT_START) -> None:
    """전체 백필. 처음에 한 번만 실행."""

    # pykrx로 실제 KRX 거래일 캘린더 조회
    start_s = start_date.strftime("%Y%m%d")
    yesterday = (date.today() - timedelta(days=1)).strftime("%Y%m%d")

    log.info("Getting trading day calendar from pykrx...")
    ohlcv = stock.get_market_ohlcv(start_s, yesterday, REF_TICKER)
    if ohlcv is None or ohlcv.empty:
        log.info("No trading days found.")
        return

    trading_days = [d.strftime("%Y%m%d") for d in ohlcv.index]
    log.info("Backfill: %d trading days (%s → %s)",
             len(trading_days), trading_days[0], trading_days[-1])

    # Phase 1: pykrx 데이터 다운로드
    all_rows = []
    for i, date_str in enumerate(trading_days, 1):
        if i % 50 == 0 or i == 1 or i == len(trading_days):
            log.info("  [%d/%d  %.0f%%] %s",
                     i, len(trading_days), i / len(trading_days) * 100, date_str)

        vol_df = fetch_short_vol(date_str)
        bal_df = fetch_short_bal(date_str)
        merged = merge_vol_bal(vol_df, bal_df, date_str)
        if not merged.empty:
            all_rows.append(merged)

        # 100일치 모이면 중간 저장 (메모리 절약 + 중간 끊김 대비)
        if len(all_rows) >= 100:
            flush_to_stocks(all_rows)
            all_rows.clear()

    if all_rows:
        flush_to_stocks(all_rows)

    log.info("Phase 1 complete — per-stock CSVs written.")

    # Phase 2: Bloomberg enrichment (optional)
    if _HAS_BBG:
        log.info("Phase 2 — Bloomberg enrichment...")
        all_tickers = [p.stem for p in sorted(REPO_DIR.glob("*.csv"))]
        px_hist, ff_hist, idx_hist = fetch_bbg_history(
            all_tickers, start_date, date.today() - timedelta(days=1))

        for ticker in all_tickers:
            df = read_stock(ticker)
            if df.empty:
                continue
            # 각 종목의 각 날짜에 대해 Bloomberg 값을 매핑
            if ticker in px_hist:
                df["px_last"] = df["date"].map(px_hist[ticker]).astype(float)
            else:
                df["px_last"] = np.nan
            if ticker in ff_hist:
                df["free_float"] = df["date"].map(ff_hist[ticker]).astype(float)
            else:
                df["free_float"] = np.nan
            if idx_hist:
                df["index_px"] = df.apply(
                    lambda r: idx_hist.get(r.get("시장"), pd.Series()).get(r["date"], np.nan),
                    axis=1).astype(float)
            else:
                df["index_px"] = np.nan
            write_stock(ticker, df)
        log.info("Bloomberg enrichment complete.")
    else:
        log.warning("Bloomberg not available — px_last/free_float/index_px will be NaN.")

    log.info("Backfill complete.")

# ── 처음 실행할 때만 아래 주석 해제 ──
# backfill()

In [ ]:
# =============================================================================
# Daily Update — 매일 실행하면 빠진 거래일만 추가
# =============================================================================
#
# 흐름:
#   1) 005930.csv에서 마지막 날짜 확인 (예: 20260430)
#   2) pykrx에 "20260501~오늘" 사이의 실제 거래일 조회
#   3) 각 거래일마다 bulk API 4 call + Bloomberg → 각 종목 CSV에 1행 append
#
# 소요: ~15초/일 (Bloomberg 없이), ~40초/일 (Bloomberg 포함)
# idempotent: 이미 있는 날짜는 절대 중복 저장 안 됨
# =============================================================================

def daily_update() -> None:
    """빠진 거래일 자동 감지 후 fetch + append."""
    REPO_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Repo의 마지막 날짜 확인
    latest = latest_repo_date()
    if latest is None:
        log.info("Repo가 비어있음 — backfill()을 먼저 실행하세요.")
        return

    log.info("Repo 마지막 날짜: %s", latest)

    # 2. 빠진 실제 거래일 목록 조회
    missing = find_missing_trading_days(latest)
    if not missing:
        log.info("이미 최신 상태입니다.")
        return

    log.info("빠진 거래일 %d일: %s → %s", len(missing), missing[0], missing[-1])

    # 3. 각 거래일 fetch + append
    for i, date_str in enumerate(missing, 1):
        log.info("  Fetching %s ...", date_str)
        d = date(int(date_str[:4]), int(date_str[4:6]), int(date_str[6:]))

        # pykrx bulk fetch (4 calls)
        vol_df = fetch_short_vol(date_str)
        bal_df = fetch_short_bal(date_str)
        merged = merge_vol_bal(vol_df, bal_df, date_str)

        if merged.empty:
            log.info("  %s: 데이터 없음 — skip", date_str)
            continue

        # Bloomberg enrichment (있으면)
        tickers_today = merged["티커"].unique().tolist()
        px_map, ff_map, idx_map = fetch_bbg_daily(tickers_today, d)
        merged["px_last"] = merged["티커"].map(px_map).astype(float)
        merged["free_float"] = merged["티커"].map(ff_map).astype(float)
        merged["index_px"] = merged["시장"].map(idx_map).astype(float)

        # 각 종목 CSV에 1행 append
        appended = 0
        for ticker, group in merged.groupby("티커"):
            row = group.drop(columns=["티커"])
            existing = read_stock(ticker)
            if existing.empty:
                write_stock(ticker, row)                    # 신규 종목
            elif date_str not in existing["date"].values:
                combined = pd.concat([existing, row], ignore_index=True)
                write_stock(ticker, combined)               # 기존 종목에 append
            # else: 이미 해당 날짜가 있으면 skip (idempotent)
            appended += 1

        log.info("  [%d/%d] %s: %d종목 (bbg: %d px, %d ff, %d idx)",
                 i, len(missing), date_str, appended,
                 len(px_map), len(ff_map), len(idx_map))

    log.info("업데이트 완료.")


# ── 실행 ──
daily_update()

In [ ]:
# =============================================================================
# Status — Repo 현황 확인
# =============================================================================

def print_status():
    if not REPO_DIR.exists():
        print("Repo가 아직 없음 — backfill()을 실행하세요.")
        return

    csvs = list(REPO_DIR.glob("*.csv"))
    latest = latest_repo_date()
    print(f"\n{'='*50}")
    print(f"Short Data Repository: {REPO_DIR.resolve()}")
    print(f"{'='*50}")
    print(f"  종목 수:    {len(csvs):,}")
    print(f"  최신 날짜:  {latest}")
    print(f"  Bloomberg:  {'available' if _HAS_BBG else 'not installed'}")

    if latest:
        missing = find_missing_trading_days(latest)
        print(f"  빠진 거래일: {len(missing)}일")

    # 삼성전자 샘플로 구조 확인
    ref = read_stock(REF_TICKER)
    if not ref.empty:
        print(f"\n  샘플 ({REF_TICKER} 삼성전자):")
        print(f"    행 수:    {len(ref)}")
        print(f"    기간:     {ref['date'].min()} → {ref['date'].max()}")
        print(f"    컬럼:     {list(ref.columns)}")

        # 마지막 3행 미리보기
        print(f"\n    최근 데이터:")
        display(ref.tail(3))
    print()

print_status()